In [ ]:
%stop_session

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore",
  "--enable-glue-datacatalog": "true"
}

In [ ]:
#Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import DeltaTable
from datetime import *
from pyspark.sql.types import *
from pyspark.sql.utils import AnalysisException

BASE = "s3://data-lake-case-hotmart"

# D-1 
RUN_DATE = (date.today() - timedelta(days=1)).isoformat() #D-1 troque para a data que deseja navegar
# uat : RUN_DATE = "2023-03-31"   # teste também: "2023-01-23", "2023-02-28", "2023-07-15"

PATH_BRONZE_PURCHASE         = f"{BASE}/bronze/purchase"
PATH_BRONZE_PRODUCT_ITEM     = f"{BASE}/bronze/product_item"
PATH_BRONZE_EXTRA_INFO       = f"{BASE}/bronze/purchase_extra_info"

PATH_SILVER_PURCHASE         = f"{BASE}/silver/purchase_daily"
PATH_SILVER_PRODUCT_ITEM     = f"{BASE}/silver/product_item_daily"
PATH_SILVER_EXTRA_INFO       = f"{BASE}/silver/purchase_extra_info_daily"

PATH_GOLD                    = f"{BASE}/gold/gmv_daily"
PATH_GOLD_GMV                = f"{BASE}/gold/gmv_daily_by_subsidiary"

### BRONZE — purchase (eventos CDC)
- Bronze é append-only.

In [ ]:

# Criar DataFrame diretamente (dados do enunciado - Exercício 2)
df_purchase_bronze = spark.createDataFrame([
    ("2023-01-20 22:00:00", "2023-01-20", 55, 15947, 5, "2023-01-20", "2023-01-20", 852852),
    ("2023-01-26 00:01:00", "2023-01-26", 56, 369798, 746520, "2023-01-25", None, 963963),
    ("2023-02-05 10:00:00", "2023-02-05", 55, 160001, 5, "2023-01-20", "2023-01-20", 852852),
    ("2023-02-26 03:00:00", "2023-02-26", 69, 160001, 18, "2023-01-26", "2023-02-28", 96967),
    ("2023-07-15 09:00:00", "2023-07-15", 55, 160001, 5, "2023-01-20", "2023-03-01", 852852)
], schema=[
    "transaction_datetime",
    "transaction_date",
    "purchase_id",
    "buyer_id",
    "prod_item_id",
    "order_date",
    "release_date",
    "producer_id"
])

# Tipos corretos
df_purchase_bronze = df_purchase_bronze \
    .withColumn("transaction_datetime", col("transaction_datetime").cast("timestamp")) \
    .withColumn("transaction_date", col("transaction_date").cast("date")) \
    .withColumn("purchase_id", col("purchase_id").cast("bigint")) \
    .withColumn("buyer_id", col("buyer_id").cast("bigint")) \
    .withColumn("prod_item_id", col("prod_item_id").cast("bigint")) \
    .withColumn("producer_id", col("producer_id").cast("bigint")) \
    .withColumn("order_date", col("order_date").cast("date")) \
    .withColumn("release_date", col("release_date").cast("date"))

# Controle de rastreabilidade
df_purchase_bronze = df_purchase_bronze.withColumn(
    "ingestion_date",
    to_utc_timestamp(current_timestamp(), "UTC")
)


# Persistir Bronze em DELTA (append-only)
df_purchase_bronze.write \
    .format("delta") \
    .mode("append") \
    .partitionBy("transaction_date") \
    .save(PATH_BRONZE_PURCHASE)

In [ ]:
# uat
df_purchase_bronze.createOrReplaceTempView("vw_bronze_purchase")

spark.sql("SELECT * FROM vw_bronze_purchase").show()